> （版本/依赖/环境）对齐，跑起来 -> 跑得对 -> 跑得快/跑得稳

- verl main commit
- use_dynamic_bsz vs. use_remove_padding
    - use_dynamic_bsz：决定每个 micro-batch 放哪些样本、放多少条样本。
    - use_remove_padding：micro-batch 确定后，决定送进模型时是否删除 padding，并把有效 token 拼成 packed tokens。
- remove-padding 把多个样本伪装成一条长序列
    - use_remove_padding=False
        - https://github.com/verl-project/verl/blob/main/examples/grpo_trainer/run_qwen3_5_2b_openr1_fsdp.sh
        - PR#6660：https://github.com/verl-project/verl/pull/6660
        - 不打任何 verl 补丁，最省心；代价是没有 packing 的吞吐收益。
```
# micro-batch 有两个样本：
A = [A0 A1 A2 A3 A4]       长度 5
B = [B0 B1 B2]             长度 3

input_ids_rmpad = [[A0 A1 A2 A3 A4 B0 B1 B2]]   shape = [1, 8]
cu_seqlens = [0, 5, 8]
```

- Qwen3.5-4B/9B 都是混合结构：
    - 32 layers = 24 × linear_attention + 8 × full_attention
- 线性注意力（linear attention）不是为当前 token 重新读取全部历史 token，而是维护一个循环状态（recurrent state）$S_t$。Gated DeltaNet 的简化状态更新可以写为：
    - $S_t=\alpha_t S_{t-1}(I-\beta_t k_tk_t^\top)+\beta_t v_tk_t^\top$
    - 并通过 $o_t=S_tq_t$ 读取。
- 旧的 Qwen3.5 Gated DeltaNet forward 没收到 cu_seqlens。它看到的只是 batch size 1、sequence length 8，于是把 A、B 当成一条连续序列。
    - 如果没有 `cu_seqlens=[0,5,8]`：
    - B0 的 causal conv 会读到 A 的最后几个 token；
    - B0 的 Gated DeltaNet 循环状态会继承 A 的最终状态；
    - FSDP 算出的 $\pi_\theta(B)$ 实际变成了受 A 污染的概率；
    - vLLM 则仍按独立样本计算 B。
    - $\pi_{\text{vLLM}}(B\mid B_{\text{prefix}})\neq\pi_{\text{FSDP}}(B\mid A,B_{\text{prefix}})$
    - https://github.com/verl-project/verl/issues/6780
- vLLM 走的不是那条有问题的 Hugging Face/FSDP packed-training forward。它虽然也会把多个请求的 token 拼在一起计算，但始终保留：
    - 每个请求的边界
    - 每个请求独立的卷积状态（convolution state）
    - 每个请求独立的 GatedDeltaNet 循环状态（recurrent state）
    - Qwen3.5 没有直接调用原始 Transformers 的 Qwen3_5GatedDeltaNet.forward，而是使用 vLLM 自己的推理实现：`v1/attention/backends/gdn_attn.py`

- ulysses_sequence_parallel_size=1(Qwen3.5 线性注意力硬性)

```
docker exec verl-qwen35 bash -c 'cd /workspace/verl-train/verl && \
  torchrun --nproc_per_node=2 -m pytest -q tests/special_distributed/test_qwen35_linear_attention_ulysses_sp.py'
```

- `train_rollout_logprob_abs_diff`: verl/slime

### 镜像与容器

- `docker run --rm --gpus all verlai/verl:vllm023.dev1 nvidia-smi`
    - https://github.com/verl-project/verl/blob/main/.github/workflows/vllm.yml#L75
        - 最新 docker image
    - pull & test nvidia-driver

#### custom dockerfile、image

```Dockerfile
# file: Dockerfile.qwen35
FROM verlai/verl:vllm023.dev1
# transformers>=5.4: 修 Qwen3.5 3D-mRoPE+FA2 越界写NaN(HF#44643), 原生识别 model_type=qwen3_5
# fla-core/flash-linear-attention 0.5.1: gated-deltanet 线性注意力快核 (禁0.5.0; bf16 only)
# TransferQueue: verl trainer v1 数据面
RUN pip install --no-cache-dir -U \
      "transformers==5.12.1" \
      "fla-core==0.5.1" "flash-linear-attention==0.5.1" \
      "TransferQueue==0.1.8"
# causal-conv1d 必须 --no-build-isolation: 否则 pip 构建隔离会用错配的 torch 编译 CUDA 扩展,
#   运行时 ABI 错位 -> causal_conv1d_fwd 在 A100 上 SIGSEGV. 此处按本镜像 torch 2.11.0+cu130 重编.
#   (已用 PR#6660 的 SP 测试验证 causal_conv1d_fn 路径通过)
RUN pip install --no-cache-dir --no-build-isolation --no-deps "causal_conv1d==1.6.2.post1"
CMD ["sleep","infinity"]
```

- `docker build -f Dockerfile.qwen35 -t verl-qwen35:v1 .`
    - 基于 dockerfile 创建 image
 
```shell
sudo docker run --rm verl-qwen35:v1 python -c \
 "import transformers,fla,causal_conv1d; print('transformers',transformers.__version__); import importlib.metadata as m; print('fla',m.version('flash-linear-attention'))"

sudo docker run --rm verl-qwen35:v1 pip list | \
  grep -iE "^transformers |^fla-core|^flash-linear|^TransferQueue|^vllm |^torch "
```

#### verl

```sh
cd /home/user/verl-train
git clone https://github.com/verl-project/verl.git
# 30119a25，git checkout xxx
cd verl && git rev-parse --short HEAD    # 记下这个 SHA，便于复现
cd /home/user/verl-train
```

#### container & verl install

```sh
sudo docker create --name verl-qwen35 \
  --init \
  --gpus all \
  --pid=host \
  --net=host \
  --shm-size=128g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  --cap-add=SYS_ADMIN \
  -v /home/user/verl-train:/workspace/verl-train \
  -v /data1/models:/workspace/models:ro \
  -v /home/user/.cache/huggingface:/root/.cache/huggingface \
  -e HF_ENDPOINT=https://hf-mirror.com \
  -e NVIDIA_DRIVER_CAPABILITIES=compute,utility \
  -e VLLM_WORKER_MULTIPROC_METHOD=spawn \
  -e TOKENIZERS_PARALLELISM=false \
  -w /workspace/verl-train \
  verl-qwen35:v1 \
  sleep infinity
```

```sh
sudo docker start verl-qwen35
```

```sh
sudo docker exec verl-qwen35 nvidia-smi -L

sudo docker exec -it verl-qwen35 bash

# editable 装 verl（只装本体、不动依赖）
sudo docker exec verl-qwen35 bash -c "cd /workspace/verl-train/verl && pip install --no-deps -e . && python -c 'import verl,transformers,fla;print(verl.__version__,transformers.__version__)'"
```

### misc

```sh
sudo docker exec -it verl-qwen35 bash
cd verl 
git config --global --add safe.directory /workspace/verl-train/verl


wandb login
# check，会将账户信息写入
~/.netrc
```

### swe-agent docker

```
docker run -d \
  --name verl-swe-agent \
  --gpus all \
  --network host \
  --shm-size=128g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -e HF_ENDPOINT=https://hf-mirror.com \
  -e VLLM_WORKER_MULTIPROC_METHOD=spawn \
  -e TOKENIZERS_PARALLELISM=false \
  -v /home/user/verl-train:/workspace/verl-train \
  -v /data1/models:/workspace/models:ro \
  -v /home/user/.cache/huggingface:/root/.cache/huggingface \
  -v /var/run/docker.sock:/var/run/docker.sock \
  -v /usr/bin/docker:/usr/bin/docker:ro \
  -w /workspace/verl-train \
  verl-qwen35:v1 \
  sleep infinity
```

- `-v /var/run/docker.sock:/var/run/docker.sock`
    - 让容器内能起 sandbox 容器
- `-v /usr/bin/docker:/usr/bin/docker:ro `
    - docker CLI，即容器内能识别 docker 命令
- 查看宿主机与容器的文件/夹挂载
    - docker inspect verl-swe-agent --format '{{range .Mounts}}{{.Source}} -> {{.Destination}} (RW={{.RW}}){{println}}{{end}}'

### slime

- 最佳方式是 Docker（slimerl/slime:latest）
    - https://github.com/THUDM/slime/blob/main/docs/zh/examples/qwen3-4B.md
 
```
docker pull slimerl/slime:latest
```

```
docker run -d --name slime-geo3k \
  --gpus all --ipc=host --shm-size=32g \
  --ulimit memlock=-1 --ulimit stack=67108864 --ulimit nofile=1048576:1048576 \
  -e HTTP_PROXY=http://172.17.0.1:7890  -e HTTPS_PROXY=http://172.17.0.1:7890 \
  -e http_proxy=http://172.17.0.1:7890  -e https_proxy=http://172.17.0.1:7890 \
  -e NO_PROXY=localhost,127.0.0.1,::1   -e no_proxy=localhost,127.0.0.1,::1 \
  -v /home/user/slime-train/slime:/root/slime \
  -v /data1/models:/root/models \
  -v /home/user/slime-train:/root/host \
  -v /home/user/slime-train/data:/root/datasets \
  -v /home/user/.netrc:/root/.netrc:ro \
  slimerl/slime:latest sleep infinity
```